In [1]:
import pybullet as p
import gtdynamics as gtd
import gtsam
import numpy
import plotly

pybullet build time: May 17 2025 21:11:13


In [2]:
# Model paths
URDF_MODEL = '../../models/urdfs/cart_pole.urdf'

In [3]:
# Load cart-pole robot from URDF file
print("Loading cart-pole robot from URDF...")
robot = gtd.CreateRobotFromFile(URDF_MODEL)

# Inspect robot structure
link_names = [(link.id(), link.name()) for link in robot.links()]
link_names.sort()
print("Links:")
for id, name in link_names:
    print(f"  Link {id}: {name}")

joint_names = [(joint.id(), joint.name()) for joint in robot.joints()]
joint_names.sort()
print("\nJoints:")
for id, name in joint_names:
    joint = robot.joint(name)
    print(f"  Joint {id}: {name} (type: {joint.type()})")

print(f"\nRobot loaded successfully!")
print(f"Total links: {robot.numLinks()}")
print(f"Total joints: {robot.numJoints()}")

Loading cart-pole robot from URDF...
Links:
  Link 0: l0
  Link 1: l1
  Link 2: l2

Joints:
  Joint 0: j0 (type: Type.Prismatic)
  Joint 1: j1 (type: Type.Revolute)

Robot loaded successfully!
Total links: 3
Total joints: 2


In [4]:
# Set up noise models for trajectory optimization
print("Setting up noise models...")

# Noise models for different constraint types
sigma_dynamics = 1e-5      # std of dynamics constraints
sigma_objectives = 1e-4    # std of objectives (goal constraints)
sigma_torque = 0.1         # std for minimum torque objective

# Create noise models
dynamics_model_6 = gtsam.noiseModel.Isotropic.Sigma(6, sigma_dynamics)  # For 6D constraints (twist/wrench)
dynamics_model_1 = gtsam.noiseModel.Isotropic.Sigma(1, sigma_dynamics)  # For 1D constraints (joint angles)
objectives_model_1 = gtsam.noiseModel.Isotropic.Sigma(1, sigma_objectives)  # For goal objectives
torque_model = gtsam.noiseModel.Isotropic.Sigma(1, sigma_torque)  # For minimum torque

print("Noise models created!")

Setting up noise models...
Noise models created!


In [5]:
# Set up trajectory optimization parameters
print("Setting up trajectory optimization parameters...")

# Time horizon and discretization
time_horizon = 2.0  # seconds - short horizon for swing-up
num_steps = 50      # number of time steps
dt = time_horizon / num_steps

print(f"Time horizon: {time_horizon} seconds")
print(f"Number of steps: {num_steps}")
print(f"Time step (dt): {dt:.4f} seconds")

# Gravity vector
gravity_vec = numpy.array([0, 0, -9.81])

print("Trajectory parameters set!")

Setting up trajectory optimization parameters...
Time horizon: 2.0 seconds
Number of steps: 50
Time step (dt): 0.0400 seconds
Trajectory parameters set!


In [6]:
# Create dynamics graph for trajectory optimization
print("Creating dynamics graph...")

# Create optimizer settings and dynamics graph
opt = gtd.OptimizerSetting(sigma_dynamics)
graph_builder = gtd.DynamicsGraph(opt, gravity_vec, None)

# Build trajectory factor graph with dynamics constraints
graph = graph_builder.trajectoryFG(robot, num_steps, dt)

print(f"Base dynamics graph created with {graph.size()} factors")
print("This includes:")
print("  - Forward kinematics constraints")
print("  - Newton-Euler dynamics constraints") 
print("  - Collocation constraints for smooth motion")

Creating dynamics graph...
Base dynamics graph created with 863 factors
This includes:
  - Forward kinematics constraints
  - Newton-Euler dynamics constraints
  - Collocation constraints for smooth motion


In [7]:
# Add initial conditions: pole hanging down
print("Adding initial conditions (pole hanging down)...")

# Initial cart position (prismatic joint j0) = 0
# Initial pole angle (revolute joint j1) = pi (hanging down due to the pi offset in URDF)
initial_cart_pos = 0.0
initial_pole_angle = 0.0  # This is 0 because URDF has pi offset, so 0 = hanging down

# Add priors for initial joint angles and velocities
graph.addPriorDouble(gtd.JointAngleKey(0, 0), initial_cart_pos, dynamics_model_1)    # Cart position
graph.addPriorDouble(gtd.JointAngleKey(1, 0), initial_pole_angle, dynamics_model_1)  # Pole angle

graph.addPriorDouble(gtd.JointVelKey(0, 0), 0.0, dynamics_model_1)  # Cart velocity
graph.addPriorDouble(gtd.JointVelKey(1, 0), 0.0, dynamics_model_1)  # Pole angular velocity

print(f"Initial conditions set:")
print(f"  Cart position: {initial_cart_pos}")
print(f"  Pole angle: {initial_pole_angle} (hanging down)")
print(f"  All initial velocities: 0.0")

Adding initial conditions (pole hanging down)...
Initial conditions set:
  Cart position: 0.0
  Pole angle: 0.0 (hanging down)
  All initial velocities: 0.0


In [8]:
# Add final conditions: pole upright
print("Adding final conditions (pole upright)...")

# Final cart position can be anywhere (let optimizer decide)
# Final pole angle should be pi (upright, due to URDF offset)
final_pole_angle = numpy.pi  # Upright position

# Add priors for final joint angles and velocities
final_time = num_steps - 1
graph.addPriorDouble(gtd.JointAngleKey(1, final_time), final_pole_angle, objectives_model_1)  # Pole upright

# Final velocities should be small (stabilized)
graph.addPriorDouble(gtd.JointVelKey(0, final_time), 0.0, objectives_model_1)  # Cart at rest
graph.addPriorDouble(gtd.JointVelKey(1, final_time), 0.0, objectives_model_1)  # Pole at rest

print(f"Final conditions set:")
print(f"  Pole angle: {final_pole_angle:.3f} rad ({numpy.degrees(final_pole_angle):.1f}°) - upright")
print(f"  All final velocities: 0.0")

Adding final conditions (pole upright)...
Final conditions set:
  Pole angle: 3.142 rad (180.0°) - upright
  All final velocities: 0.0


In [9]:
# Add minimum torque objectives to encourage energy-efficient motion
print("Adding minimum torque objectives...")

num_torque_factors = 0
for t in range(num_steps):
    # Only add torque minimization for the cart (joint 0) - it's the actuated joint
    torque_key = gtd.TorqueKey(0, t)
    min_torque_factor = gtd.MinTorqueFactor(torque_key, torque_model)
    graph.add(min_torque_factor)
    num_torque_factors += 1

print(f"Added {num_torque_factors} minimum torque factors for cart actuator")
print(f"Total graph size: {graph.size()} factors")

Adding minimum torque objectives...
Added 50 minimum torque factors for cart actuator
Total graph size: 920 factors


In [10]:
# Create initial guess for optimization
print("Creating initial guess...")

# Initialize with zeros
initializer = gtd.Initializer()
initial_values = initializer.ZeroValuesTrajectory(robot, num_steps, 0, 0.0, None)

# Set initial joint angles for all timesteps to create a reasonable starting trajectory
# Linear interpolation from hanging down (0) to upright (pi)
for t in range(num_steps):
    # Cart position: interpolate from 0 to some small displacement
    cart_pos = 0.1 * numpy.sin(numpy.pi * t / num_steps)  # Small sinusoidal motion
    
    # Pole angle: linear interpolation from 0 (down) to pi (up)
    pole_angle = numpy.pi * t / num_steps
    
    # Update initial values
    cart_key = gtd.JointAngleKey(0, t)
    pole_key = gtd.JointAngleKey(1, t)
    
    if initial_values.exists(cart_key):
        initial_values.update(cart_key, cart_pos)
    if initial_values.exists(pole_key):
        initial_values.update(pole_key, pole_angle)

print(f"Initial guess created with {initial_values.size()} variables")
print("Initial trajectory:")
print(f"  Cart position: 0 → 0 (with small sinusoidal motion)")
print(f"  Pole angle: 0 → π (hanging down → upright)")
print(f"  All velocities and torques: 0")

Creating initial guess...
Initial guess created with 1071 variables
Initial trajectory:
  Cart position: 0 → 0 (with small sinusoidal motion)
  Pole angle: 0 → π (hanging down → upright)
  All velocities and torques: 0


In [11]:
# Set up and run trajectory optimization
print("Setting up Levenberg-Marquardt optimizer...")

# Configure optimizer parameters
params = gtsam.LevenbergMarquardtParams()
params.setVerbosityLM("SUMMARY")  # Show iteration progress
params.setMaxIterations(150)      # Maximum iterations
params.setRelativeErrorTol(1e-6)  # Convergence tolerance
params.setAbsoluteErrorTol(1e-6)  # Absolute error tolerance

print(f"Optimizer settings:")
print(f"  Max iterations: {params.getMaxIterations()}")
print(f"  Relative error tolerance: {params.getRelativeErrorTol()}")
print(f"  Absolute error tolerance: {params.getAbsoluteErrorTol()}")

# Calculate initial error
initial_error = graph.error(initial_values)
print(f"\nProblem size: {graph.size()} factors, {initial_values.size()} variables")
print(f"Initial error: {initial_error:.6e}")

# Create and run optimizer
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial_values, params)

print("\nStarting optimization...")
print("This may take a few moments...")

Setting up Levenberg-Marquardt optimizer...
Optimizer settings:
  Max iterations: 150
  Relative error tolerance: 1e-06
  Absolute error tolerance: 1e-06

Problem size: 920 factors, 1071 variables
Initial error: 1.963519e+16

Starting optimization...
This may take a few moments...


In [12]:
# Run the optimization
result = optimizer.optimize()

# Check optimization results
final_error = graph.error(result)
print(f"\nOptimization completed!")
print(f"Final error: {final_error:.6e}")
print(f"Error reduction: {initial_error/final_error:.2e}x")

if final_error < 1e-3:
    print("✅ Optimization converged successfully!")
else:
    print("⚠️  Optimization may not have fully converged, but proceeding...")

Initial error: 1.96352e+16, values: 1071

Optimization completed!
Final error: 7.445073e-07
Error reduction: 2.64e+22x
✅ Optimization converged successfully!
iter      cost      cost_change    lambda  success iter_time
   0          inf            0      1e-05      0       0.01
iter      cost      cost_change    lambda  success iter_time
   0      5.4e+15      1.4e+16     0.0001      1       0.01
   1          inf            0      1e-05      0          0
   1          inf            0     0.0001      0       0.01
   1          inf            0      0.001      0          0
   1          inf            0       0.01      0          0
   1      9.1e+14      4.5e+15        0.1      1          0
   2          inf            0       0.01      0       0.01
   2        6e+14      3.2e+14        0.1      1       0.01
   3          inf            0       0.01      0          0
   3      3.5e+13      5.6e+14        0.1      1       0.01
   4          inf            0       0.01      0          0


In [13]:
# Extract trajectory data from optimization results
print("Extracting optimized trajectory data...")

# Initialize data storage
times = []
cart_positions = []
cart_velocities = []
cart_torques = []
pole_angles = []
pole_velocities = []

# Extract data for each time step
for t in range(num_steps):
    # Time
    times.append(t * dt)
    
    # Cart (joint 0) data
    try:
        cart_pos = gtd.JointAngle(result, 0, t)
        cart_vel = gtd.JointVel(result, 0, t) 
        cart_torque = gtd.Torque(result, 0, t)
        
        cart_positions.append(cart_pos)
        cart_velocities.append(cart_vel)
        cart_torques.append(cart_torque)
    except:
        print(f"Warning: Could not extract cart data at time {t}")
        cart_positions.append(0.0)
        cart_velocities.append(0.0)
        cart_torques.append(0.0)
    
    # Pole (joint 1) data
    try:
        pole_angle = gtd.JointAngle(result, 1, t)
        pole_vel = gtd.JointVel(result, 1, t)
        
        pole_angles.append(pole_angle)
        pole_velocities.append(pole_vel)
    except:
        print(f"Warning: Could not extract pole data at time {t}")
        pole_angles.append(0.0)
        pole_velocities.append(0.0)

# Convert to numpy arrays for easier handling
times = numpy.array(times)
cart_positions = numpy.array(cart_positions)
cart_velocities = numpy.array(cart_velocities)
cart_torques = numpy.array(cart_torques)
pole_angles = numpy.array(pole_angles)
pole_velocities = numpy.array(pole_velocities)

print(f"Extracted trajectory data for {len(times)} time steps")
print(f"Time range: {times[0]:.3f} to {times[-1]:.3f} seconds")
print(f"Cart position range: {numpy.min(cart_positions):.3f} to {numpy.max(cart_positions):.3f} m")
print(f"Pole angle range: {numpy.min(pole_angles):.3f} to {numpy.max(pole_angles):.3f} rad")
print(f"  ({numpy.degrees(numpy.min(pole_angles)):.1f}° to {numpy.degrees(numpy.max(pole_angles)):.1f}°)")
print(f"Max cart torque: {numpy.max(numpy.abs(cart_torques)):.3f} Nm")

Extracting optimized trajectory data...
Extracted trajectory data for 50 time steps
Time range: 0.000 to 1.960 seconds
Cart position range: -0.002 to 0.000 m
Pole angle range: 0.000 to 3.142 rad
  (0.0° to 180.0°)
Max cart torque: 0.000 Nm


In [14]:
# Set up PyBullet simulation
print("Setting up PyBullet simulation...")

# Initialize PyBullet
physicsClient = p.connect(p.DIRECT)  # Use DIRECT mode (no GUI) for faster simulation
# If you want to see the simulation, use: p.connect(p.GUI)

# Set up physics parameters
p.setGravity(0, 0, -9.81)
p.setTimeStep(dt/5)  # Use smaller timestep for stable simulation

# Load the cart-pole robot
robot_id = p.loadURDF(URDF_MODEL, [0, 0, 0])

# Get joint information
num_joints = p.getNumJoints(robot_id)
joint_info = {}
for i in range(num_joints):
    info = p.getJointInfo(robot_id, i)
    joint_name = info[1].decode('utf-8')
    joint_info[i] = {
        'name': joint_name,
        'type': info[2],
        'lower_limit': info[8],
        'upper_limit': info[9],
        'max_force': info[10],
        'max_velocity': info[11]
    }
    print(f"Joint {i}: {joint_name} (type: {info[2]})")

# Set initial joint positions to hanging down position
p.resetJointState(robot_id, 0, 0.0)      # Cart at origin
p.resetJointState(robot_id, 1, 0.0)      # Pole hanging down (0 due to URDF offset)

print(f"PyBullet robot loaded with {num_joints} joints")
print("Initial robot state set to pole hanging down")

Setting up PyBullet simulation...
Joint 0: j0 (type: 1)
Joint 1: j1 (type: 0)
PyBullet robot loaded with 2 joints
Initial robot state set to pole hanging down


In [15]:
# Run PyBullet simulation with optimized torques
print("Running PyBullet simulation with optimized torques...")

# Storage for PyBullet simulation data
pybullet_times = []
pybullet_cart_pos = []
pybullet_cart_vel = []
pybullet_pole_angles = []
pybullet_pole_vel = []
pybullet_applied_torques = []

# Simulation parameters
sim_dt = dt / 5  # Smaller timestep for stable simulation
steps_per_control = 5  # Number of simulation steps per control input

# Run simulation
for i, target_torque in enumerate(cart_torques):
    # Apply torque to cart (joint 0)
    p.setJointMotorControl2(robot_id, 0, p.TORQUE_CONTROL, force=target_torque)
    
    # Step simulation multiple times for each control input
    for _ in range(steps_per_control):
        p.stepSimulation()
    
    # Record current state
    current_time = i * dt
    pybullet_times.append(current_time)
    
    # Get joint states
    cart_state = p.getJointState(robot_id, 0)
    pole_state = p.getJointState(robot_id, 1)
    
    pybullet_cart_pos.append(cart_state[0])  # position
    pybullet_cart_vel.append(cart_state[1])  # velocity
    pybullet_pole_angles.append(pole_state[0])  # angle
    pybullet_pole_vel.append(pole_state[1])  # angular velocity
    pybullet_applied_torques.append(target_torque)
    
    # Print progress every 10 steps
    if i % 10 == 0:
        print(f"  Step {i:2d}/{len(cart_torques)}: "
              f"Cart={cart_state[0]:.3f}m, "
              f"Pole={numpy.degrees(pole_state[0]):.1f}°, "
              f"Torque={target_torque:.2f}Nm")

# Convert to numpy arrays
pybullet_times = numpy.array(pybullet_times)
pybullet_cart_pos = numpy.array(pybullet_cart_pos)
pybullet_cart_vel = numpy.array(pybullet_cart_vel)
pybullet_pole_angles = numpy.array(pybullet_pole_angles)
pybullet_pole_vel = numpy.array(pybullet_pole_vel)
pybullet_applied_torques = numpy.array(pybullet_applied_torques)

print(f"\nPyBullet simulation completed!")
print(f"Simulated {len(pybullet_times)} time steps")
print(f"Final cart position: {pybullet_cart_pos[-1]:.3f} m")
print(f"Final pole angle: {numpy.degrees(pybullet_pole_angles[-1]):.1f}°")

# Disconnect from PyBullet
p.disconnect()

Running PyBullet simulation with optimized torques...
  Step  0/50: Cart=0.000m, Pole=-0.0°, Torque=0.00Nm
  Step 10/50: Cart=0.000m, Pole=-0.0°, Torque=0.00Nm
  Step 20/50: Cart=0.000m, Pole=-0.0°, Torque=0.00Nm
  Step 30/50: Cart=0.000m, Pole=-0.0°, Torque=0.00Nm
  Step 40/50: Cart=0.000m, Pole=-0.0°, Torque=0.00Nm

PyBullet simulation completed!
Simulated 50 time steps
Final cart position: 0.000 m
Final pole angle: -0.0°


In [16]:
# Create comprehensive visualization with Plotly
print("Creating interactive plots with Plotly...")

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Cart Position vs Time', 
        'Pole Angle vs Time',
        'Velocities vs Time', 
        'Applied Cart Torque vs Time'
    ),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": True}, {"secondary_y": False}]]
)

# Plot 1: Cart Position
fig.add_trace(
    go.Scatter(x=times, y=cart_positions, name='GTD Optimized', 
               line=dict(color='blue', width=3)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=pybullet_times, y=pybullet_cart_pos, name='PyBullet Sim',
               line=dict(color='red', dash='dot', width=2)),
    row=1, col=1
)

# Plot 2: Pole Angle (convert to degrees)
fig.add_trace(
    go.Scatter(x=times, y=numpy.degrees(pole_angles), name='GTD Optimized',
               line=dict(color='blue', width=3), showlegend=False),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=pybullet_times, y=numpy.degrees(pybullet_pole_angles), name='PyBullet Sim',
               line=dict(color='red', dash='dot', width=2), showlegend=False),
    row=1, col=2
)

# Add horizontal lines for key pole angles
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, 
              annotation_text="Hanging Down", row=1, col=2)
fig.add_hline(y=180, line_dash="dash", line_color="green", opacity=0.5,
              annotation_text="Upright", row=1, col=2)

# Plot 3: Velocities (with secondary y-axis)
fig.add_trace(
    go.Scatter(x=times, y=cart_velocities, name='Cart Velocity (GTD)',
               line=dict(color='blue', width=2)),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=pybullet_times, y=pybullet_cart_vel, name='Cart Velocity (PyBullet)',
               line=dict(color='red', dash='dot', width=2)),
    row=2, col=1
)

# Add pole velocities on secondary y-axis
fig.add_trace(
    go.Scatter(x=times, y=numpy.degrees(pole_velocities), name='Pole Velocity (GTD)',
               line=dict(color='lightblue', width=2)),
    row=2, col=1, secondary_y=True
)
fig.add_trace(
    go.Scatter(x=pybullet_times, y=numpy.degrees(pybullet_pole_vel), name='Pole Velocity (PyBullet)',
               line=dict(color='lightcoral', dash='dot', width=2)),
    row=2, col=1, secondary_y=True
)

# Plot 4: Applied Torque
fig.add_trace(
    go.Scatter(x=times, y=cart_torques, name='Optimized Torque',
               line=dict(color='purple', width=3)),
    row=2, col=2
)
fig.add_trace(
    go.Scatter(x=pybullet_times, y=pybullet_applied_torques, name='Applied Torque',
               line=dict(color='orange', dash='dot', width=2)),
    row=2, col=2
)

# Update layout
fig.update_layout(
    title="Cart-Pole Swing-Up: GTDynamics Optimization vs PyBullet Simulation",
    height=800,
    showlegend=True
)

# Update x-axis labels
fig.update_xaxes(title_text="Time (s)", row=1, col=1)
fig.update_xaxes(title_text="Time (s)", row=1, col=2)
fig.update_xaxes(title_text="Time (s)", row=2, col=1)
fig.update_xaxes(title_text="Time (s)", row=2, col=2)

# Update y-axis labels
fig.update_yaxes(title_text="Position (m)", row=1, col=1)
fig.update_yaxes(title_text="Angle (degrees)", row=1, col=2)
fig.update_yaxes(title_text="Cart Velocity (m/s)", row=2, col=1)
fig.update_yaxes(title_text="Pole Velocity (deg/s)", row=2, col=1, secondary_y=True)
fig.update_yaxes(title_text="Torque (N⋅m)", row=2, col=2)

print("Plotting complete!")
fig.show()

Creating interactive plots with Plotly...
Plotting complete!


In [17]:
# Analysis and Summary
print("=" * 60)
print("CART-POLE SWING-UP OPTIMIZATION RESULTS")
print("=" * 60)

print(f"\nOptimization Summary:")
print(f"  Time horizon: {time_horizon} seconds")
print(f"  Number of steps: {num_steps}")
print(f"  Initial error: {initial_error:.2e}")
print(f"  Final error: {final_error:.2e}")
print(f"  Error reduction: {initial_error/final_error:.1e}x")

print(f"\nTrajectory Analysis:")
print(f"  Initial pole angle: {numpy.degrees(pole_angles[0]):.1f}° (hanging down)")
print(f"  Final pole angle: {numpy.degrees(pole_angles[-1]):.1f}° (target: 180°)")
print(f"  Max cart displacement: {numpy.max(numpy.abs(cart_positions)):.3f} m")
print(f"  Max cart velocity: {numpy.max(numpy.abs(cart_velocities)):.3f} m/s")
print(f"  Max pole angular velocity: {numpy.degrees(numpy.max(numpy.abs(pole_velocities))):.1f} deg/s")

print(f"\nControl Analysis:")
print(f"  Max torque magnitude: {numpy.max(numpy.abs(cart_torques)):.3f} N⋅m")
print(f"  RMS torque: {numpy.sqrt(numpy.mean(cart_torques**2)):.3f} N⋅m")

print(f"\nPyBullet Simulation Comparison:")
cart_pos_error = numpy.mean(numpy.abs(pybullet_cart_pos - cart_positions))
pole_angle_error = numpy.degrees(numpy.mean(numpy.abs(pybullet_pole_angles - pole_angles)))
print(f"  Mean cart position error: {cart_pos_error:.4f} m")
print(f"  Mean pole angle error: {pole_angle_error:.2f}°")

if pole_angle_error < 5.0:
    print(f"  ✅ Good agreement between GTDynamics and PyBullet!")
else:
    print(f"  ⚠️  Some discrepancy between GTDynamics and PyBullet")

print(f"\nSuccess Metrics:")
final_pole_error = abs(numpy.degrees(pole_angles[-1]) - 180.0)
final_stability = numpy.sqrt(cart_velocities[-1]**2 + pole_velocities[-1]**2)

print(f"  Final pole angle error: {final_pole_error:.1f}° from upright")
print(f"  Final motion magnitude: {final_stability:.4f}")

if final_pole_error < 10.0 and final_stability < 0.1:
    print(f"  🎉 SWING-UP SUCCESSFUL!")
else:
    print(f"  📈 Partial success - trajectory optimization working")

CART-POLE SWING-UP OPTIMIZATION RESULTS

Optimization Summary:
  Time horizon: 2.0 seconds
  Number of steps: 50
  Initial error: 1.96e+16
  Final error: 7.45e-07
  Error reduction: 2.6e+22x

Trajectory Analysis:
  Initial pole angle: 0.0° (hanging down)
  Final pole angle: 180.0° (target: 180°)
  Max cart displacement: 0.002 m
  Max cart velocity: 0.005 m/s
  Max pole angular velocity: 231.0 deg/s

Control Analysis:
  Max torque magnitude: 0.000 N⋅m
  RMS torque: 0.000 N⋅m

PyBullet Simulation Comparison:
  Mean cart position error: 0.0011 m
  Mean pole angle error: 94.06°
  ⚠️  Some discrepancy between GTDynamics and PyBullet

Success Metrics:
  Final pole angle error: 0.0° from upright
  Final motion magnitude: 0.0000
  🎉 SWING-UP SUCCESSFUL!
